In [1]:
from pathlib import Path

# نام پروژه
project_name = "iran-war-market-analysis"

# ساختار فولدرها
folders = [
    f"{project_name}",
    f"{project_name}/data/raw",
    f"{project_name}/data/external",
    f"{project_name}/data/processed",
    f"{project_name}/notebooks",
    f"{project_name}/src",
    f"{project_name}/visuals",
    f"{project_name}/reports",
]

# ساخت فولدرها
for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)
    print(f"Created: {folder}")

print("Project structure created successfully!")

Created: iran-war-market-analysis
Created: iran-war-market-analysis/data/raw
Created: iran-war-market-analysis/data/external
Created: iran-war-market-analysis/data/processed
Created: iran-war-market-analysis/notebooks
Created: iran-war-market-analysis/src
Created: iran-war-market-analysis/visuals
Created: iran-war-market-analysis/reports
Project structure created successfully!


In [2]:
pip install yfinance pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [3]:
import yfinance as yf
import pandas as pd

oil = yf.download("BZ=F", start="2024-01-01", end="2026-05-20")

[*********************100%***********************]  1 of 1 completed


In [4]:
gold = yf.download("GC=F", start="2024-01-01", end="2026-05-20")

[*********************100%***********************]  1 of 1 completed


In [5]:
eurusd = yf.download("EURUSD=X", start="2024-01-01", end="2026-05-20")

[*********************100%***********************]  1 of 1 completed


In [10]:
import os

print(os.getcwd())
print(os.path.exists("data"))
print(os.path.exists("data/external"))
print(type(oil))
print(oil.shape)
print(oil.head())

c:\git-projects
True
True
<class 'pandas.DataFrame'>
(599, 5)
Price           Close       High        Low       Open Volume
Ticker           BZ=F       BZ=F       BZ=F       BZ=F   BZ=F
Date                                                         
2024-01-02  75.889999  79.040001  75.599998  77.209999  28591
2024-01-03  78.250000  78.669998  74.790001  75.989998  32172
2024-01-04  77.589996  79.400002  76.500000  78.500000  33154
2024-01-05  78.760002  79.250000  77.500000  77.599998  30994
2024-01-08  76.120003  78.949997  75.260002  78.599998  32539


In [11]:
import pandas as pd

def fix_yahoo(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.droplevel(1)
    return df

In [12]:
oil = fix_yahoo(oil)
gold = fix_yahoo(gold)
eurusd = fix_yahoo(eurusd)

In [13]:
print(oil.columns)
print(gold.columns)
print(eurusd.columns)

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='str', name='Price')
Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='str', name='Price')
Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='str', name='Price')


In [14]:
oil.to_csv("data/external/oil.csv")
gold.to_csv("data/external/gold.csv")
eurusd.to_csv("data/external/eurusd.csv")

In [15]:
oil = oil[["Close"]]
gold = gold[["Close"]]
eurusd = eurusd[["Close"]]

In [16]:
oil_close = oil["Close"].rename("oil")
gold_close = gold["Close"].rename("gold")
eurusd_close = eurusd["Close"].rename("eurusd")

In [17]:
import pandas as pd

df = pd.concat([oil_close, gold_close, eurusd_close], axis=1)
df = df.dropna()
print(df.head())
print(df.info())


                  oil         gold    eurusd
Date                                        
2024-01-02  75.889999  2064.399902  1.103875
2024-01-03  78.250000  2034.199951  1.094176
2024-01-04  77.589996  2042.300049  1.092777
2024-01-05  78.760002  2042.400024  1.094739
2024-01-08  76.120003  2026.599976  1.094224
<class 'pandas.DataFrame'>
DatetimeIndex: 598 entries, 2024-01-02 to 2026-05-19
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   oil     598 non-null    float64
 1   gold    598 non-null    float64
 2   eurusd  598 non-null    float64
dtypes: float64(3)
memory usage: 18.7 KB
None


C:\Users\esmae\AppData\Local\Temp\ipykernel_45360\4220239480.py:3: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([oil_close, gold_close, eurusd_close], axis=1)


In [18]:
df_norm = df / df.iloc[0] * 100

In [24]:
import os

os.chdir(r"c:\git-projects\my-first-project\iran-war-market-analysis")
print(os.getcwd())

c:\git-projects\my-first-project\iran-war-market-analysis


In [25]:
df.to_csv("data/processed/market_merged.csv")
df_norm.to_csv("data/processed/market_normalized.csv")

In [26]:
import os

BASE_DIR = r"c:\git-projects\my-first-project\iran-war-market-analysis"

processed_path = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(processed_path, exist_ok=True)

df.to_csv(os.path.join(processed_path, "market_merged.csv"))

In [29]:
import pandas as pd
import os

BASE = r"c:\git-projects\my-first-project\iran-war-market-analysis"

market = pd.read_csv(
    os.path.join(BASE, "data/processed/market_merged.csv"),
    index_col=0,
    parse_dates=True
)

timeline = pd.read_csv(
    os.path.join(BASE, "data/external/war/timeline.csv")
)

timeline["date"] = pd.to_datetime(timeline["date"])
timeline = timeline.sort_values("date")

In [ ]:
market["war_event"] = 0

In [30]:
for d in timeline["date"]:
    if d in market.index:
        market.loc[d, "war_event"] = 1

In [32]:
master = market.copy()

master[["oil", "gold", "eurusd"]] = master[["oil", "gold", "eurusd"]].pct_change()

master = master.dropna()
print(master.head())
print(master.info())    



                 oil      gold    eurusd  war_event
Date                                               
2025-06-13  0.070213  0.014878  0.007900        1.0
2025-06-24 -0.060716 -0.017852  0.008537        1.0
2026-02-24 -0.010071 -0.009395 -0.003456        1.0
2026-03-02  0.072572  0.012217 -0.003692        1.0
2026-03-04  0.000000  0.002506 -0.007269        1.0
<class 'pandas.DataFrame'>
DatetimeIndex: 13 entries, 2025-06-13 to 2026-04-17
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   oil        13 non-null     float64
 1   gold       13 non-null     float64
 2   eurusd     13 non-null     float64
 3   war_event  13 non-null     float64
dtypes: float64(4)
memory usage: 520.0 bytes
None


In [33]:
master.to_csv(os.path.join(BASE, "data/processed/master_dataset.csv"))

In [35]:
import os

os.chdir(r"C:\git-projects\my-first-project\iran-war-market-analysis")
print(os.getcwd())

C:\git-projects\my-first-project\iran-war-market-analysis


In [37]:
import pandas as pd
import os

# =========================
# BASE PATH (FIXED)
# =========================
base_path = "data/external/war"

# =========================
# LOAD DATASETS
# =========================
timeline = pd.read_csv(f"{base_path}/timeline.csv")
economic = pd.read_csv(f"{base_path}/economic_impact.csv")
casualties = pd.read_csv(f"{base_path}/casualties.csv", usecols=["Date"])
infra = pd.read_csv(f"{base_path}/infrastructure_damage.csv")
aircraft = pd.read_csv(f"{base_path}/aircraft_losses.csv")
naval = pd.read_csv(f"{base_path}/naval_losses.csv")

# =========================
# DATE FIX
# =========================
for df in [timeline, economic, casualties, infra, aircraft, naval]:
    df["Date"] = pd.to_datetime(df["Date"])

# =========================
# MERGE MASTER DATASET
# =========================
df = timeline.copy()

df = df.merge(economic, on="Date", how="outer")
df = df.merge(casualties, on="Date", how="outer")
df = df.merge(infra, on="Date", how="outer")
df = df.merge(aircraft, on="Date", how="outer")
df = df.merge(naval, on="Date", how="outer")

df = df.sort_values("Date")

# =========================
# SAVE OUTPUT
# =========================
os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/master_dataset.csv", index=False)

print("DONE: Master dataset created")

ValueError: Usecols do not match columns, columns expected but not found: ['Date']

In [39]:
import os

os.remove("data/external/war/casualties.csv")
print("deleted")

deleted


In [40]:
import yfinance as yf

oil = yf.download("BZ=F", start="2024-01-01")
gold = yf.download("GC=F", start="2024-01-01")
eurusd = yf.download("EURUSD=X", start="2024-01-01")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [42]:
oil = oil.reset_index()
gold = gold.reset_index()
eurusd = eurusd.reset_index()
market = oil[["Date","Close"]].rename(columns={"Close":"Oil"})
market = market.merge(gold[["Date","Close"]].rename(columns={"Close":"Gold"}), on="Date", how="outer")
market = market.merge(eurusd[["Date","Close"]].rename(columns={"Close":"EURUSD"}), on="Date", how="outer")

In [58]:
import os
import pandas as pd

base_path = r"C:\git-projects\my-first-project\iran-war-market-analysis\data\external"

war_path = f"{base_path}\war"
market_path = f"{base_path}\market"
timeline = pd.read_csv(f"{war_path}/timeline.csv")
economic = pd.read_csv(f"{war_path}/economic_impact.csv")
infra = pd.read_csv(f"{war_path}/infrastructure_damage.csv")
aircraft = pd.read_csv(f"{war_path}/aircraft_losses.csv")
naval = pd.read_csv(f"{war_path}/naval_losses.csv") 


In [62]:
def inspect_csv(folder_path):
    files = os.listdir(folder_path)
    
    for file in files:
        if file.endswith(".csv"):
            path = os.path.join(folder_path, file)
            
            print("\n" + "="*50)
            print("FILE:", file)
            
            df = pd.read_csv(path)
            
            print("SHAPE:", df.shape)
            print("COLUMNS:", df.columns.tolist())
            print("HEAD:")
            print(df.head(6))
inspect_csv(war_path)   
inspect_csv(market_path)    




FILE: aircraft_losses.csv
SHAPE: (26, 7)
COLUMNS: ['date', 'side', 'aircraft_type', 'quantity', 'loss_type', 'location', 'notes']
HEAD:
         date  side                         aircraft_type quantity  \
0  2026-02-28  Iran                Elbit Hermes 900 Drone        1   
1  2026-03-01  Iran                          Northrop F-5        1   
2  2026-03-01  Iran      McDonnell Douglas F-4 Phantom II        1   
3  2026-03-01  Iran                          Sukhoi Su-24        2   
4  2026-03-01    US  McDonnell Douglas F-15E Strike Eagle        3   
5  2026-03-02  Iran                          Sukhoi Su-22        2   

             loss_type                         location  \
0            Shot down         Khamaneh East Azerbaijan   
1  Destroyed on ground     Tabriz Shahid Madani Airport   
2  Destroyed on ground     Tabriz Shahid Madani Airport   
3            Shot down           Near Al Udeid Air Base   
4            Shot down                           Kuwait   
5  Destroyed on gr

In [82]:
import pandas as pd

# =========================
# PATH
# =========================
base_path = r"C:\git-projects\my-first-project\iran-war-market-analysis\data\external\war"

# =========================
# LOAD DATA
# =========================
timeline = pd.read_csv(f"{base_path}/timeline.csv")
infra = pd.read_csv(f"{base_path}/infrastructure_damage.csv")

# =========================
# STANDARDIZE COLUMNS
# =========================
timeline.columns = timeline.columns.str.lower()
infra.columns = infra.columns.str.lower()

# =========================
# DATE PARSING
# =========================
timeline["date"] = pd.to_datetime(timeline["date"], errors="coerce")
infra["date"] = pd.to_datetime(infra["date"], errors="coerce")

# =========================
# CLEAN TIMELINE
# =========================
timeline_clean = (
    timeline[["date", "event", "category", "significance"]]
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

# =========================
# CLEAN INFRASTRUCTURE
# =========================
infra_clean = (
    infra[['date', 'attacking_side', 'target_country', 'target_name', 'target_type', 'status', 'notes;;']]
    .rename(columns={'notes;;': 'notes'})
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

# =========================
# OUTPUT CHECK
# =========================
print("TIMELINE CLEAN:")
print(timeline_clean.head())

print("\nINFRASTRUCTURE CLEAN:")
print(infra_clean.head())
output_path = r"C:\git-projects\my-first-project\iran-war-market-analysis\data\processed"

timeline_clean.to_csv(f"{output_path}/timeline_clean.csv", index=False)
infra_clean.to_csv(f"{output_path}/infra_clean.csv", index=False)

print("Files saved successfully!")

TIMELINE CLEAN:
        date                                              event    category  \
0 2025-06-13        Israel launches Twelve-Day War against Iran    Military   
1 2025-06-22             US strikes three Iranian nuclear sites    Military   
2 2025-06-24                           Twelve-Day War ceasefire  Diplomatic   
3 2026-01-01                        Mass Iranian protests begin   Political   
4 2026-02-24  Trump State of Union claims Iran rebuilding nukes   Political   

                                        significance  
0  Israel attacks 100+ Iranian targets including ...  
1  US joins conflict; Iran fires missiles at US b...  
2            Israel-Iran ceasefire under US pressure  
3  Largest protests since 1979; Iranian regime cr...  
4  Trump claims Iran restarting nuclear program; ...  

INFRASTRUCTURE CLEAN:
        date attacking_side target_country               target_name  \
0 2026-02-28      US/Israel           Iran   Natanz Nuclear Facility   
1 2026-02-28

In [87]:
import pandas as pd

base_path = r"C:\git-projects\my-first-project\iran-war-market-analysis\data\external\war"
output_path = r"C:\git-projects\my-first-project\iran-war-market-analysis\data\processed"

# =========================
# LOAD
# =========================
timeline = pd.read_csv(f"{base_path}/timeline.csv")
infra = pd.read_csv(f"{base_path}/infrastructure_damage.csv")

# =========================
# LOWERCASE COLUMNS
# =========================
timeline.columns = timeline.columns.str.lower()
infra.columns = infra.columns.str.lower()

# =========================
# STANDARD DATE
# =========================
timeline["date"] = pd.to_datetime(timeline["date"], errors="coerce")
infra["date"] = pd.to_datetime(infra["date"], errors="coerce")

# =========================
# NORMALIZE TIMELINE
# =========================
timeline_clean = timeline.rename(columns={
    "event": "description"
})

timeline_clean["type"] = "timeline"

timeline_clean = timeline_clean[[
    "date", "type", "category", "significance", "description"
]]

timeline_clean = timeline_clean.rename(columns={
    "significance": "severity"
})

# =========================
# NORMALIZE INFRASTRUCTURE
# =========================
infra_clean = infra.rename(columns={
    "damage_type": "category",
    "notes": "description"
})

infra_clean["type"] = "infrastructure"

infra_clean = infra.rename(columns={
    "target_type": "category",
    "notes;;": "description"
})
infra_clean["type"] = "infrastructure"
infra_clean["severity"] = infra_clean["status"]
infra_clean = infra_clean[[
    "date", "type", "category", "severity", "description"
]]

# =========================
# ALIGN SCHEMAS
# =========================
common_cols = ["date", "type", "category", "severity", "description"]

timeline_clean = timeline_clean[common_cols]
infra_clean = infra_clean[common_cols]

# =========================
# UNION (STACK ROWS)
# =========================
war_master = pd.concat([timeline_clean, infra_clean], ignore_index=True)

# =========================
# SORT
# =========================
war_master = war_master.sort_values("date").reset_index(drop=True)

# =========================
# SAVE
# =========================
war_master.to_csv(f"{output_path}/war_master_dataset.csv", index=False)

print(war_master.head())
print("\nSaved successfully!")

        date      type    category  \
0 2025-06-13  timeline    Military   
1 2025-06-22  timeline    Military   
2 2025-06-24  timeline  Diplomatic   
3 2026-01-01  timeline   Political   
4 2026-02-24  timeline   Political   

                                            severity  \
0  Israel attacks 100+ Iranian targets including ...   
1  US joins conflict; Iran fires missiles at US b...   
2            Israel-Iran ceasefire under US pressure   
3  Largest protests since 1979; Iranian regime cr...   
4  Trump claims Iran restarting nuclear program; ...   

                                         description  
0        Israel launches Twelve-Day War against Iran  
1             US strikes three Iranian nuclear sites  
2                           Twelve-Day War ceasefire  
3                        Mass Iranian protests begin  
4  Trump State of Union claims Iran rebuilding nukes  

Saved successfully!


In [94]:
import pandas as pd
import os

# =========================
# 1. BASE PATH
# =========================
base_path = r"C:\git-projects\my-first-project\iran-war-market-analysis"

raw_path = os.path.join(base_path, "data", "external", "market")
out_path = os.path.join(base_path, "data", "processed")

os.makedirs(out_path, exist_ok=True)

# =========================
# 2. LOAD DATA
# =========================
oil = pd.read_csv(os.path.join(raw_path, "oil.csv"))
gold = pd.read_csv(os.path.join(raw_path, "gold.csv"))
eurusd = pd.read_csv(os.path.join(raw_path, "eurusd.csv"))

# =========================
# 3. STANDARDIZE FUNCTION
# =========================
def clean_market(df):
    # اگر ستون‌ها MultiIndex باشن (مثل Yahoo)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.rename(columns={
        "Adj Close": "Close"
    })

    # فقط ستون‌های اصلی
    df = df[["Date", "Close", "High", "Low", "Open", "Volume"]]

    # تاریخ
    df["Date"] = pd.to_datetime(df["Date"])

    # فیلتر زمانی
    df = df[df["Date"] >= "2025-04-01"]

    return df.sort_values("Date")


# =========================
# 4. CLEAN ALL MARKETS
# =========================
oil = clean_market(oil)
gold = clean_market(gold)
eurusd = clean_market(eurusd)

# =========================
# 5. SAVE CLEAN DATA
# =========================
oil.to_csv(os.path.join(out_path, "oil_clean.csv"), index=False)
gold.to_csv(os.path.join(out_path, "gold_clean.csv"), index=False)
eurusd.to_csv(os.path.join(out_path, "eurusd_clean.csv"), index=False)

print("DONE ✔ cleaned market data saved")

DONE ✔ cleaned market data saved
